In [ ]:
from rbg.DecisionTree import RegressionTree, BaggingReg, RandomForestRegression

import numpy as np 


import altair as alt
import pandas as pd

from rbg.Utilities import MSE, train_test_split

from rbg.ModelSelection import GridSearch, KFold




Faremos agora o uso de árvores para prever os nossos dados. Primeiro, faremos uma separação de dados 
de treino, e dados de teste. Em seguida, aplicaremos às árvores de regressão com e sem poda, bagging e Random
Forest os nossos dados, afim de que tenhamos uma previsão. Começaremos primeiro com os dados todos usando
os valores padrão, a fim de ilustrar o uso da biblioteca, e também  faremos apenas um plot gráfico comparativo entre a árvore com poda, e sem poda e tiraremos o erro entre essas árvores. Além do erro calculado com o MSE, também faremos o OOB da random e bagging tree. 






Validação: 

Para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, o melhor hiperparâmetro para eles. 

Comparação de Modelos:

Com os modelos ajustados pelos seus "hiperparâmetros mais ajustados", iremos fazer uma validação cruzada cmo os dados de treino para decidir qual o melhor modelo. 

Avaliação Final:

Dado o modelo escolhido, iremos fazer a predição dos dados de teste, e tomar seu erro. 

Interpretação dos resultados:

Parte destinada a mais considerações sobre escolhas de métricas e trade-off do viés e da variância.

 

In [ ]:
def plot_arvore(tree, feature_names=None):

    info = tree.get_tree_structure_info()

    left = info['children_left']
    right = info['children_right']
    feature = info['feature']
    threshold = info['threshold']
    value = info['value']

    pos = {}
    proxima_folha = [0]

    def posicao(node, depth=0):
        filhos = [c for c in (left[node], right[node]) if c != -1]
        if not filhos:
            x = proxima_folha[0]
            proxima_folha[0] += 1
        else:
            xs = [posicao(c, depth + 1) for c in filhos]
            x = sum(xs) / len(xs)
        pos[node] = (x, -depth)
        return x

    posicao(0)


    nos = []
    for node, (x, y) in pos.items():
        is_leaf = (left[node] == -1 and right[node] == -1)

        if is_leaf:
            val = value[node]
            if isinstance(val, (np.ndarray, list)):
                pred = f"{np.round(val, 2)}"
            else:
                pred = f"{val:.2f}" if isinstance(val, (int, float, np.number)) else f"{val}"
            nos.append({"x": x, "y": y, "texto": f"Pred: {pred}", "pred": pred, "folha": True})
        else:
            feat_idx = feature[node]
            if feature_names is not None and feat_idx < len(feature_names):
                feat_name = feature_names[feat_idx]
            else:
                feat_name = f"X[{feat_idx}]"
            texto = f"{feat_name} ≤ {threshold[node]:.2f}"
            nos.append({"x": x, "y": y, "texto": texto, "pred": "", "folha": False})
    nos = pd.DataFrame(nos)
    arestas = []
    for node, (x, y) in pos.items():
        for child in (left[node], right[node]):
            if child != -1:
                x2, y2 = pos[child]
                arestas.append({"x": x, "y": y, "x2": x2, "y2": y2})
    arestas = pd.DataFrame(arestas)

    eixo_x = alt.X("x:Q", axis=None, scale=alt.Scale(padding=60))
    eixo_y = alt.Y("y:Q", axis=None, scale=alt.Scale(padding=30))

    linhas = alt.Chart(arestas).mark_rule(color="gray").encode(
        x=eixo_x, y=eixo_y, x2="x2:Q", y2="y2:Q"
    )

    decisoes = alt.Chart(nos).transform_filter(
        alt.datum.folha == False
    ).mark_square(size=300, stroke="black", color="lightgray", opacity=1).encode(
        x=eixo_x, y=eixo_y, tooltip=["texto"]
    )

    folhas = alt.Chart(nos).transform_filter(
        alt.datum.folha == True
    ).mark_square(size=300, stroke="black", opacity=1).encode(
        x=eixo_x, y=eixo_y,
        color=alt.Color("pred:N", title="Pred"),
        tooltip=["texto"],
    )

    rotulos = alt.Chart(nos).mark_text(dy=-18, fontSize=11).encode(
        x=eixo_x, y=eixo_y, text="texto"
    )

    n_folhas = proxima_folha[0]
    return (linhas + decisoes + folhas + rotulos).properties(
        width=max(400, n_folhas * 110),
        height=max(250, (-nos["y"].min() + 1) * 90),
        title="Visualização da Estrutura da Árvore de Decisão",
    ).configure_view(stroke=None).interactive()

In [4]:
df = pd.read_csv('dataset_limpo.csv') 

colunas_X = ['area','yearc','location','bath','kitchen','cheating','district']

X = df[colunas_X].to_numpy()

y = df['rent'].to_numpy()



In [5]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X,y,random_state = 42) 

In [6]:
arvore = RegressionTree()

arvore.fit(X_treino,y_treino)

y_previsto = arvore.predict(X_teste)

erro_arvore_sem_poda = MSE(y_previsto,y_teste)

print("erro da arvore sem poda:",erro_arvore_sem_poda)

erro da arvore sem poda: 38932.53483366935


In [7]:
arvore_poda = RegressionTree()

arvore_poda.fit(X_treino,y_treino, prune = True)

y_previsto_poda = arvore_poda.predict(X_teste)

erro_arvore_com_poda = MSE(y_previsto_poda,y_teste)

print("erro da arvore com poda:",erro_arvore_com_poda)

info = arvore_poda.get_tree_structure_info()

erro da arvore com poda: 18992.242473276678


In [8]:
arvorebag = BaggingReg()

arvorebag.fit(X_treino,y_treino)

y_previstobag = arvorebag.predict(X_teste)

erro_arvore_bag = MSE(y_previstobag,y_teste)

print("erro da bagging :",erro_arvore_bag)

erro da bagging : 20658.106670739562


In [9]:
arvore_random = RandomForestRegression()

arvore_random.fit(X_treino,y_treino)

y_previsto_random = arvore_random.predict(X_teste)

erro_arvore_random = MSE(y_previsto_random,y_teste)

print("erro da arvore random:",erro_arvore_random)

erro da arvore random: 17998.1641147183


Plot da árvore com poda:

In [10]:
preditores = ['area', 'yearc', 'location', 'bath', 'kitchen', 'cheating', 'district']

# Chama a função de plotagem:
plot_arvore(arvore_poda, feature_names=preditores)

alt.LayerChart(...)

Plot da árvore sem poda:

In [11]:
preditores = ['area', 'yearc', 'location', 'bath', 'kitchen', 'cheating', 'district']

# Chama a função de plotagem:
plot_arvore(arvore, feature_names=preditores)

alt.LayerChart(...)

Notavelmente, se o que mais desejamos é interpretabilidade, então uma árvore com poda nesse cenário parece ser a melhor opção. 

# Validação:

## Validação da Árvore:

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) tem um custo grande para serem montadas de acordo com o algoritmo implementado (a escolha de melhor splitter que passa por todos os valores, etc), e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada, usaremos apenas os preditores:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split (e o nosso "criterion_evaluation" será por tanto igual ao criterion_split)
- criterion_choose 

Dentro de criterion_choose, também iremos apenas avaliar a média e a mediana, já que preços contínuos não costumam ter moda, e também deixaremos de fora "min" e "max" pois problemas de estimar preço, em geral será ou de fato, a mediana, ou a média, já que "min" e "max" são justamente "valores de extremo". 

Também, nos limitaremos a apenas MSE, RMSE e MAE para serem os critérios de cálculo de impureza. 





In [12]:
#Parametros:

p_grid = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2,10,50],
    'min_impurity_decrease':[0.01, 0.1, 1,10],
    'criterion_split':[ 'Standard','RMSE','MAE'], 
    'criterion_choose':['Standard','median']

}

grid_arvore = GridSearch( model = RegressionTree(), param_grid = p_grid, metrics = MSE, cv = KFold(k = 5, random_state = 42) )

In [13]:
grid_arvore.fit(X_treino,y_treino)

Agora iremos plotar os parâmetros escolhidos pela nossa árvore.

In [14]:
print(grid_arvore.best_params_)

{'max_depth': 5, 'min_samples_split': 50, 'min_impurity_decrease': 0.01, 'criterion_split': 'RMSE', 'criterion_choose': 'median'}


Para árvores do tipo baggin, que por terem sido implementadas apenas com numpy (logo, sem a biblioteca de paralelização), fazem as árvores sequencialmente. Logo, fazer apenas uma árvore de bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda. Mudaremos apenas pro bagging a quantidade de árvores utilizadas. Também faremos uma busca de hiperparâmetros dos elementos visto anteriormente, mas com ainda menos elementos, para ser viável computacionalmente. 

In [15]:
pbaggin_grid = {
    'max_depth': [10],
    'min_samples_split': [10],
    'min_impurity_decrease':[1,10],
    'criterion_split':[ 'Standard','RMSE'], 
    'criterion_choose':['Standard','median'],
    'n_trees':[10, 20]

}

grid_bagging = GridSearch( model = BaggingReg(), param_grid = pbaggin_grid, metrics = MSE, cv = KFold(k = 5, random_state = 42) )

In [16]:
grid_bagging.fit(X_treino,y_treino)

In [17]:
print(grid_bagging.best_params_)

{'max_depth': 10, 'min_samples_split': 10, 'min_impurity_decrease': 1, 'criterion_split': 'Standard', 'criterion_choose': 'Standard', 'n_trees': 20}


Similarmete, também vamos testar, para random forest, apenas 20,30 árvores e com o critério de escolha de quantidade de parâmetros ser piso da raiz quadrada do número total de preditores. Como a nosso dataset tem apenas 7 preditores, temos que sempre dois preditores serão escolhidos. 

In [26]:

p_random_grid = {
    'max_depth': [10],
    'min_samples_split': [10],
    'min_impurity_decrease':[1,10],
    'criterion_split':[ 'Standard','RMSE'], 
    'criterion_choose':['Standard','median'],
    'n_trees':[10, 20],
     'factor_selection': [np.sqrt]

}
grid_RandomReg = GridSearch( model = BaggingReg(), param_grid = p_random_grid, metrics = MSE, cv = KFold(k = 5, random_state = 42) )



In [27]:
grid_RandomReg.fit(X_treino,y_treino)

# Seleção de Modelo


Para a seleção de modelos, iremos fazer um K-Fold usando "os campeões dos hiperparâmetros", treinaremos o dataset neles, e escolheremos aquele que teve menor erro de acordo com o MSE

In [28]:

param_r = grid_RandomReg.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

print(param_r)

{'max_depth': 10, 'min_samples_split': 10, 'min_impurity_decrease': 1, 'criterion_split': 'Standard', 'criterion_choose': 'Standard', 'n_trees': 20, 'factor_selection': <ufunc 'sqrt'>}


In [29]:
param_r = grid_RandomReg.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

baggingReg = BaggingReg(**param_bagging)
arvore_pos_grid = RegressionTree(**param_arvore)
randomReg = RandomForestRegression(**param_r)

In [30]:



k_fold = KFold(random_state = 41, k = 5)
k = k_fold.split(X_treino.copy())
lista_erros_arvore =[] 
lista_erros_bag=[] 
lista_erros_random =[] 


for _, (treino_k, teste_k) in enumerate(k):
    X_treino_k, X_teste_k = X_treino[treino_k], X_treino[teste_k]
    y_treino_k, y_teste_k = y_treino[treino_k], y_treino[teste_k]

    randomReg.fit(X_treino_k,y_treino_k)
    baggingReg.fit(X_treino_k,y_treino_k)
    arvore_pos_grid.fit(X_treino_k,y_treino_k)
    y_pred_random = randomReg.predict(X_teste_k)
    y_pred_bagging = baggingReg.predict(X_teste_k)
    y_pred_arvore = arvore_pos_grid.predict(X_teste_k)
    lista_erros_arvore.append(MSE(y_teste_k, y_pred_arvore))
    lista_erros_bag.append(MSE(y_teste_k, y_pred_bagging)) 
    lista_erros_random.append(MSE(y_teste_k, y_pred_random))  

erro_medio_a = np.mean(lista_erros_arvore)
erro_medio_b = np.mean(lista_erros_bag)
erro_medio_r = np.mean(lista_erros_random)

    
print("erro medio da arvore:",erro_medio_a)
print("erro medio do bagging:",erro_medio_b)
print("erro medio do random forest:",erro_medio_r)

erro medio da arvore: 34045.38086494045
erro medio do bagging: 20037.99402723696
erro medio do random forest: 20577.0746602242


# Avaliação Final

Com o nosso modelo ganhador, iremos treinar todos os dados de treino, para aplicar aos nossos dados de teste,
e depois tirar o erro com o MSE 

In [31]:
if erro_medio_b <= erro_medio_a and erro_medio_b <= erro_medio_r:
    baggingReg.fit(X_treino,y_treino)
    print("bagging")
    predicao_final = baggingReg.predict(X_teste)
elif erro_medio_a <= erro_medio_b and erro_medio_a <= erro_medio_r:
    arvore_pos_grid.fit(X_treino,y_treino)
    predicao_final = arvore_pos_grid.predict(X_teste)
    print("arvores")
else:
    randomReg.fit(X_treino,y_treino)
    predicao_final = randomReg.predict(X_teste)
    print("RandomReg")

print("previsao final é:",predicao_final)

print("Erro do modelo:",MSE(predicao_final,y_teste))
    

bagging
previsao final é: [ 313.21447024  289.79660593  384.61975391  750.64506344  467.7814966
  324.68490623  308.30127217  327.14147696  641.04606646  179.33067979
  471.49804377  494.90538232  348.13282058  460.57714134  324.45631279
  621.72841788  440.35069722  435.8703344   347.09382327  438.75136073
  217.23303821  403.20038027  458.24780411  599.40019402  469.37600357
  493.31371113  443.13349481  311.67979793  770.64771605  770.16022055
  433.97952755  461.91176334  321.35270605  499.4342341   300.01959321
  570.13739948  674.07369899  531.97335077  236.17766472  466.83837754
  613.94688413  491.89446262  448.41816731  504.75308379  654.07647079
  434.11053611  347.71863643  424.607941    887.14294959  294.27750614
  309.61474331  391.83089034  314.95413748  268.02801767  352.62048603
  477.50666126  387.99638666  465.58454445  533.83604088  551.96762284
  688.05032632  621.36418161  350.55003574  261.72380887  306.37989746
  496.21498577  280.2263299   500.67878977  554.3804

Como o modelo final também foi um que tem o método OOB, podemos plotar esse erro OOB:

In [24]:


oob = randomReg.OOB()

In [25]:
print(oob)

150.0934233887926


previsao final é: [315.64102173 280.12820435 383.38461304 731.12820435 464.56410217
 315.64102173 315.64102173 315.64102173 596.9230957  213.64102936
 464.56410217 404.30770874 383.38461304 464.56410217 315.64102173
 532.61535645 464.56410217 464.56410217 306.46151733 404.30770874
 178.4102478  482.89744568 464.56410217 532.61535645 464.56410217
 464.56410217 464.56410217 313.33334351 596.9230957  842.20513916
 417.02563477 464.56410217 315.64102173 383.38461304 280.12820435
 464.56410217 596.9230957  532.61535645 297.10256958 464.56410217
 596.9230957  464.56410217 464.56410217 464.56410217 596.9230957
 383.38461304 315.64102173 464.56410217 842.20513916 297.10256958
 306.46151733 343.89741516 315.64102173 306.46151733 306.46151733
 464.56410217 383.38461304 482.89744568 590.71795654 482.89744568
 652.56408691 596.9230957  383.38461304 280.12820435 315.64102173
 464.56410217 306.46151733 711.69235229 590.71795654 658.46154785
 883.84616089 596.9230957  464.56410217 464.56410217 383.38

# Interpretção dos Resultados:



O resultado original esperado seria naturalmente que o random forest tivesse sido escolhido, justamente por ele estar associado a diminuir a variância por causa da sua seletividade de mudança de parâmetros. 

No entanto, a baixa possibilidade de para árvores devido ao custo computacional( 20 e 30), assim como o fato que originalmente tinhamos apenas 7 preditores ( e a raiz arredondada para baixo nos deixa com ), são fatores que provavelmente influenciaram o bagging ser "o vencedor" em relação ao random nessas aplicações. Caso fosse paralelizável, e pudessemos ter mais árvores, talvez random ganhasse de bagging regression. 


Em relação ao "bagging ganhar de árvores normais sem poda" é muito compreensível haja visto que, apesar das árvores sem poda terem o menor viés ( pois são as mais propícias a overfittarem), elas tem, por outro lado do trade-off viés variânvia, uma variância maior, aumentando o seu erro. 

Tanto o GridSearch quanto os modelos de árvore permitiam outras métricas, que poderiam ser exploradas pelo usuário, a depender dos critérios de decisão dele. 


Um adendo, é que poderia ter sido aplicada uma poda na árvore campeã, e mesmo assim ela retornaria resultados com um erro maior que o bagging, como pode ser visto abaixo. Como a poda foi planejada para ser um parâmetro do fit, ela não foi encaminhada como um dos parâmetros do grid. 


In [36]:



k_fold = KFold(random_state = 41, k = 5)
k = k_fold.split(X_treino.copy())
lista_erros_arvore =[] 
lista_erros_bag=[] 


for _, (treino_k, teste_k) in enumerate(k):
    X_treino_k, X_teste_k = X_treino[treino_k], X_treino[teste_k]
    y_treino_k, y_teste_k = y_treino[treino_k], y_treino[teste_k]

    
    baggingReg.fit(X_treino_k,y_treino_k)
    arvore_pos_grid.fit(X_treino_k,y_treino_k,prune = True)
    y_pred_bagging = baggingReg.predict(X_teste_k)
    y_pred_arvore = arvore_pos_grid.predict(X_teste_k)
    lista_erros_arvore.append(MSE(y_teste_k, y_pred_arvore))
    lista_erros_bag.append(MSE(y_teste_k, y_pred_bagging)) 


erro_medio_a = np.mean(lista_erros_arvore)
erro_medio_b = np.mean(lista_erros_bag)

    
print("erro medio da arvore com poda:",erro_medio_a)
print("erro medio do bagging:",erro_medio_b)


erro medio da arvore com poda: 40976.33255247613
erro medio do bagging: 20037.99402723696
